# Building a Model Quality Test Suite: From Unit Tests to Drift Detection

**Dataset:** `sklearn.datasets.load_breast_cancer` — real diagnostic measurements (fine-needle aspirate cell nuclei features) from the Wisconsin Diagnostic Breast Cancer dataset, used to classify tumors as malignant or benign.

**Problem statement:** You've just joined the ML platform team as a quality engineer, and you've inherited a tumor-classification model that a data scientist trained last quarter and shipped to production. Nobody has written a single automated check on it. Your SQA instincts tell you this is exactly the kind of system that fails silently: a model that looked fine in a notebook six months ago, now quietly degrading on a subpopulation nobody is watching, or being fed a malformed batch from an upstream pipeline change. Your job is to treat this model the way you'd treat any other piece of software under test: write an automated test suite around it.

**What you should produce:** a set of four test functions — each one callable independently, each returning a structured pass/fail verdict plus diagnostic detail (not just `True`/`False`) — that together form a pre-deployment gate for this model:

1. `validate_schema(df)` — a **data contract test**: checks that an incoming batch of feature data matches the expectations the model was trained on (no nulls, all expected columns present, every numeric column within a plausible range based on the training distribution).
2. `evaluate_slice(model, X, y, slice_mask, metric_fn, threshold)` — a **slice-based regression test**: computes a metric (e.g. recall) on a specific subpopulation (e.g. only the largest tumors) and fails if it drops below a threshold. A model can look fine in aggregate while failing badly on a slice that matters.
3. `perturbation_invariance_test(model, X, epsilon, n_samples)` — a **stability test**: checks that small, realistic perturbations to the input features (within measurement noise) don't flip the model's predicted class. A model that flips its answer for a 1% jitter in a measurement is not trustworthy.
4. `detect_drift(reference, current, alpha)` — a **distribution drift test**: uses a two-sample statistical test (Kolmogorov–Smirnov) to flag when an incoming batch's feature distribution has shifted significantly from the training distribution — the single most common reason a healthy model quietly becomes an unhealthy one in production.

Run all four against the trained model and report a single pass/fail test report, the way a CI pipeline would.


## Setup

Run this cell as-is — it loads the real dataset, splits it, and trains the baseline model your test suite will check.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score
from scipy.stats import ks_2samp

RANDOM_STATE = 42

data = load_breast_cancer(as_frame=True)
X_full = data.frame.drop(columns=["target"])
y_full = data.frame["target"]  # 0 = malignant, 1 = benign

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.25, random_state=RANDOM_STATE, stratify=y_full
)

model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
model.fit(X_train, y_train)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Holdout accuracy: {model.score(X_test, y_test):.3f}")


## Building a "production batch" to test against

The dataset has no timestamps, so to exercise your drift detector you'll compare two **real** subpopulations instead of inventing fake data: tumors with below-median `mean radius` ("reference/training-like") vs. at-or-above-median `mean radius` ("current/incoming batch"). This mimics a real scenario — e.g. a new clinic sending in patients with a different size profile than the original training hospital — using only real measurements.

You'll also want one column slightly corrupted (to exercise `validate_schema`) and one *severely* perturbed column (to exercise `detect_drift` catching a real shift) — build those from the real data below.

In [ ]:
median_radius = X_full["mean radius"].median()
reference_mask = X_full["mean radius"] < median_radius
current_mask = ~reference_mask

reference_batch = X_full[reference_mask].reset_index(drop=True)
current_batch = X_full[current_mask].reset_index(drop=True)

# TODO: make a copy of `current_batch` called `bad_batch` that simulates a broken
# upstream pipeline: introduce a few null values into the "mean texture" column
# (e.g. set 5 random rows to np.nan) to exercise validate_schema().
bad_batch = None


## Test 1 — Data contract / schema validation

Fill in `validate_schema`. It should check, against the training data's own schema (`X_train`):
- every expected column is present
- no column has null values
- every numeric column's values fall within `[train_min - buffer, train_max + buffer]`, where `buffer` is some tolerance (e.g. 10% of the training range) — a few out-of-range rows is a red flag, not necessarily a crash.

Return a dict like `{"passed": bool, "issues": [list of human-readable strings]}`.

In [ ]:
def validate_schema(df: pd.DataFrame, reference_df: pd.DataFrame = X_train, tolerance: float = 0.10) -> dict:
    \"\"\"Check df against reference_df's schema: columns, nulls, and plausible ranges.\"\"\"
    issues = []
    # TODO: check for missing/extra columns vs reference_df

    # TODO: check for null values per column

    # TODO: check each numeric column is within [min - tolerance*range, max + tolerance*range]

    return {"passed": len(issues) == 0, "issues": issues}


## Test 2 — Slice-based regression test

Fill in `evaluate_slice`. Given a trained model, features `X`, true labels `y`, a boolean `slice_mask` selecting a subpopulation, a `metric_fn` (e.g. `recall_score`), and a `threshold`, it should compute the metric **only on the sliced rows** and report pass/fail.

Then use it to check: does recall for **malignant** cases (the class where a miss is dangerous) stay above 0.90 specifically among the largest tumors (`mean radius` above the 75th percentile)? A model can hit 95% overall recall while missing most malignant cases in one risky slice — that's exactly the kind of regression aggregate metrics hide.

In [ ]:
def evaluate_slice(model, X: pd.DataFrame, y: pd.Series, slice_mask: pd.Series,
                    metric_fn, threshold: float, **metric_kwargs) -> dict:
    \"\"\"Compute metric_fn(y_true, y_pred) restricted to rows where slice_mask is True.\"\"\"
    # TODO: subset X and y using slice_mask
    # TODO: predict on the subset
    # TODO: compute the metric and compare to threshold
    # return {"passed": ..., "metric_value": ..., "n_rows": ..., "threshold": threshold}
    pass


# TODO: build a slice_mask over X_test for "mean radius" above its 75th percentile (computed on X_test)
large_tumor_mask = None

# TODO: call evaluate_slice to check recall for the malignant class (label 0) on that slice, threshold=0.90
# hint: recall_score needs pos_label=0 since 0 = malignant here
slice_result = None
print(slice_result)


## Test 3 — Perturbation invariance test

Fill in `perturbation_invariance_test`. Sample `n_samples` rows from `X`, add small Gaussian noise (`epsilon` standard deviations relative to each feature's own std) to each sampled row, and check what fraction of predictions change. A trustworthy model should be stable under noise far smaller than real measurement error.

Return something like `{"passed": bool, "flip_rate": float, "n_samples": int}`, failing if `flip_rate` exceeds some tolerance (e.g. 0.02).

In [ ]:
def perturbation_invariance_test(model, X: pd.DataFrame, epsilon: float = 0.05,
                                  n_samples: int = 100, flip_tolerance: float = 0.02,
                                  random_state: int = RANDOM_STATE) -> dict:
    \"\"\"Check that predictions don't flip under small per-feature Gaussian noise.\"\"\"
    rng = np.random.default_rng(random_state)
    # TODO: sample n_samples rows from X (rng.choice on the index)
    # TODO: get original predictions on those rows
    # TODO: add noise: for each column, noise ~ N(0, epsilon * column.std()), add to sampled rows
    # TODO: get predictions on the perturbed rows
    # TODO: compute flip_rate = fraction of predictions that changed
    # return {"passed": flip_rate <= flip_tolerance, "flip_rate": flip_rate, "n_samples": n_samples}
    pass


perturbation_result = None  # TODO: call perturbation_invariance_test(model, X_test)
print(perturbation_result)


## Test 4 — Distribution drift detection

Fill in `detect_drift`. For a single feature column, run `scipy.stats.ks_2samp(reference, current)` and flag drift if the p-value is below `alpha`. Then write `detect_drift_report` to run this across *every* feature column between `reference_batch` and `current_batch`, and separately between `reference_batch` and `bad_batch`, and compare the two reports.

Expect: comparing `reference_batch` vs `current_batch` (the real median-radius split) should show drift on `mean radius` and anything correlated with it (e.g. `mean area`, `mean perimeter`) — that's the real signal you engineered by splitting on size. Features unrelated to size should mostly *not* drift.

In [ ]:
def detect_drift(reference: pd.Series, current: pd.Series, alpha: float = 0.05) -> dict:
    \"\"\"Two-sample KS test for a single feature; flags drift if p-value < alpha.\"\"\"
    # TODO: run ks_2samp(reference, current)
    # TODO: return {"passed": p_value >= alpha, "statistic": ..., "p_value": ..., "alpha": alpha}
    pass


def detect_drift_report(reference_df: pd.DataFrame, current_df: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    \"\"\"Run detect_drift across every shared numeric column; return a summary DataFrame.\"\"\"
    # TODO: for each column in reference_df, call detect_drift and collect results into rows
    # return pd.DataFrame(rows).sort_values("p_value")
    pass


drift_report_real = None   # TODO: detect_drift_report(reference_batch, current_batch)
drift_report_bad = None    # TODO: detect_drift_report(reference_batch, bad_batch) — should look clean, bad_batch has nulls not drift
print(drift_report_real)


## Putting it together: the test report

Write one final cell that runs all four tests and prints a CI-style summary: test name, PASSED/FAILED, and the key diagnostic number for each.

In [ ]:
# TODO: assemble and print a summary report, e.g.:
# Test: validate_schema (bad_batch)      -> FAILED (3 issues)
# Test: evaluate_slice (large tumors)    -> PASSED (recall=0.94, threshold=0.90)
# Test: perturbation_invariance          -> PASSED (flip_rate=0.01)
# Test: detect_drift (mean radius)       -> FAILED (p=0.0002 < 0.05)


---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

**Approach:**
- `validate_schema` diffs columns against the reference schema, checks `.isnull().sum()` per column, and compares each column's `[min, max]` to the training range expanded by a tolerance buffer.
- `evaluate_slice` is just "filter, predict, score" — the only trick is remembering `pos_label=0` for malignant since sklearn's `recall_score` defaults to treating `1` as positive.
- `perturbation_invariance_test` adds noise scaled to each feature's *own* std so a feature measured in the hundreds and one measured in the tenths get proportionally comparable jitter.
- `detect_drift` is a thin wrapper around `ks_2samp`; the report function just loops columns and sorts by p-value so the most-drifted features sort to the top.

```python
def validate_schema(df: pd.DataFrame, reference_df: pd.DataFrame = X_train, tolerance: float = 0.10) -> dict:
    issues = []
    missing_cols = set(reference_df.columns) - set(df.columns)
    extra_cols = set(df.columns) - set(reference_df.columns)
    if missing_cols:
        issues.append(f"Missing columns: {sorted(missing_cols)}")
    if extra_cols:
        issues.append(f"Unexpected columns: {sorted(extra_cols)}")

    shared_cols = set(reference_df.columns) & set(df.columns)
    for col in shared_cols:
        n_nulls = df[col].isnull().sum()
        if n_nulls > 0:
            issues.append(f"Column '{col}' has {n_nulls} null value(s)")

        col_min, col_max = reference_df[col].min(), reference_df[col].max()
        buffer = tolerance * (col_max - col_min)
        lo, hi = col_min - buffer, col_max + buffer
        out_of_range = df[col].dropna()
        out_of_range = out_of_range[(out_of_range < lo) | (out_of_range > hi)]
        if len(out_of_range) > 0:
            issues.append(f"Column '{col}' has {len(out_of_range)} value(s) outside [{lo:.2f}, {hi:.2f}]")

    return {"passed": len(issues) == 0, "issues": issues}


def evaluate_slice(model, X, y, slice_mask, metric_fn, threshold, **metric_kwargs):
    X_slice, y_slice = X[slice_mask], y[slice_mask]
    y_pred = model.predict(X_slice)
    metric_value = metric_fn(y_slice, y_pred, **metric_kwargs)
    return {
        "passed": metric_value >= threshold,
        "metric_value": metric_value,
        "n_rows": len(X_slice),
        "threshold": threshold,
    }

large_tumor_mask = X_test["mean radius"] >= X_test["mean radius"].quantile(0.75)
slice_result = evaluate_slice(model, X_test, y_test, large_tumor_mask, recall_score,
                               threshold=0.90, pos_label=0)


def perturbation_invariance_test(model, X, epsilon=0.05, n_samples=100, flip_tolerance=0.02,
                                  random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    idx = rng.choice(X.index, size=min(n_samples, len(X)), replace=False)
    sample = X.loc[idx]
    original_preds = model.predict(sample)

    noise = pd.DataFrame(
        {col: rng.normal(0, epsilon * X[col].std(), size=len(sample)) for col in X.columns},
        index=sample.index,
    )
    perturbed = sample + noise
    perturbed_preds = model.predict(perturbed)

    flip_rate = float((original_preds != perturbed_preds).mean())
    return {"passed": flip_rate <= flip_tolerance, "flip_rate": flip_rate, "n_samples": len(sample)}

perturbation_result = perturbation_invariance_test(model, X_test)


def detect_drift(reference, current, alpha=0.05):
    stat, p_value = ks_2samp(reference.dropna(), current.dropna())
    return {"passed": p_value >= alpha, "statistic": stat, "p_value": p_value, "alpha": alpha}

def detect_drift_report(reference_df, current_df, alpha=0.05):
    shared_cols = [c for c in reference_df.columns if c in current_df.columns]
    rows = []
    for col in shared_cols:
        result = detect_drift(reference_df[col], current_df[col], alpha=alpha)
        rows.append({"feature": col, **result})
    return pd.DataFrame(rows).sort_values("p_value")

drift_report_real = detect_drift_report(reference_batch, current_batch)
drift_report_bad = detect_drift_report(reference_batch, bad_batch)
```

**What you should see:** `drift_report_real` flags `mean radius`, `mean perimeter`, `mean area` (and similar size-correlated features) with very small p-values — real, honest drift caused by the size-based split. `drift_report_bad` should look mostly clean on distribution (a handful of nulls don't shift a KS statistic much), which is precisely why schema validation and drift detection are *complementary* checks, not substitutes for each other: nulls are a contract violation, not a distributional shift, and `validate_schema` is the one that should catch `bad_batch`.

**Why this matters for ML quality engineering:** every one of these four tests maps onto a traditional software-testing concept — schema validation is a contract/interface test, slice evaluation is a regression test scoped to a risk area, perturbation testing is a stability/fuzz test, and drift detection is closest to a canary/monitoring check that runs continuously rather than once at release. The shift from SQA to ML quality work is less "learn entirely new skills" and more "learn which statistical tool stands in for the assertion you'd normally write by hand."

</details>